## **Mount Drive**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## **Importing Libraries**

In [ ]:
pip install rouge

In [ ]:
import os
import re
import cv2
import tarfile
import datetime
import warnings
import numpy as np
import pandas as pd
from PIL import Image
import seaborn as sns
from tqdm import tqdm
import tensorflow as tf
from tensorflow import concat
from tensorflow import repeat
from collections import Counter
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
warnings.filterwarnings('ignore')
from sklearn.utils import shuffle
import xml.etree.ElementTree as ET
from keras.models import load_model
from skimage.transform import resize
import nltk.translate.bleu_score as bleu
from tensorflow.keras.models import Model
from google.colab.patches import cv2_imshow
from keras.preprocessing.text import Tokenizer
from tensorflow.keras.backend import expand_dims 
from keras.layers import concatenate, Concatenate
from nltk.translate.bleu_score import sentence_bleu
from tensorflow.keras.layers import TimeDistributed
from sklearn.model_selection import train_test_split
from tensorflow.keras.applications import DenseNet121
from keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.image import img_to_array
from tensorflow.keras.applications.densenet import preprocess_input
from tensorflow.keras.applications.inception_v3 import InceptionV3, preprocess_input
from tensorflow.keras.layers import Input, Softmax, RNN, Dense, Embedding, LSTM, Layer, Dropout
from rouge import Rouge

## **Structured Data**

In [ ]:
dataframe = pd.read_csv('/content/drive/MyDrive/B.E. Project Resources/data.csv',index_col=False)

In [ ]:
projections = pd.read_csv('/content/drive/MyDrive/B.E. Project Resources/indiana_projections (1).csv')

In [ ]:
projections.head()

### **Multiple DataPoints**

In [ ]:
def front_lateral(lst):
    ''' This Function check the image projection of the different images
    and return frontal or lateral of the given image '''
    frontal = []
    lateral = ""
    for img in lst:
        projection = projections[projections['filename'].str.contains(re.search(r"\d.*\_IM-\d.*\.", img).group())]['projection'].values
        if projection == "Lateral":
            lateral = img
        else:
            frontal.append(img)
    return frontal, lateral

In [ ]:
from tqdm import tqdm

In [ ]:
columns = ["front X-Ray", "lateral X-Ray", "findings"]

df = pd.DataFrame(columns = columns)
#Creating diffrent dataframe for single images, 
#to distribute it in all train test validate dataset in equal proportion.
SingleImages_df = pd.DataFrame(columns = columns)

for item in tqdm(dataframe.iterrows()):
    img_lst = item[1]['image_name'].split(',')

    if len(img_lst) > 2:
        #check projection of x-ray images
        frontal, lateral = front_lateral(img_lst)
        
        #If no lateral x-ray is present
        if lateral == "":
            frontal, lateral = frontal[:-1], frontal[-1]
        
        #If more than one frontal image is present 
        for i in frontal:
            xray_front = i
            xray_lat = lateral
            df = df.append(pd.Series([xray_front, xray_lat, item[1]['findings']], index = columns), ignore_index = True)

    elif len(img_lst) == 2:
        xray_front = img_lst[0] 
        xray_lat = img_lst[1]
        df = df.append(pd.Series([xray_front, xray_lat, item[1]['findings']], index = columns), ignore_index = True)

    elif len(img_lst) == 1:
        xray_front = img_lst[0] 
        xray_lat = img_lst[0]
        SingleImages_df = SingleImages_df.append(pd.Series([xray_front, xray_lat, item[1]['findings']], index = columns), ignore_index = True)


In [ ]:
SingleImages_df.head()

## **Train,Test and Validation Data**

In [ ]:
# Splitting df
train1, test1  = train_test_split(df, test_size = 0.10, random_state = 42, shuffle = True)
train1, valid1 = train_test_split(train1, test_size = 0.10, random_state = 42, shuffle = True)

print('train data :', train1.shape)
print('test data  :', test1.shape)
print('valid data :', valid1.shape)

In [ ]:
# Splitting SingleImages_df
train2, test2  = train_test_split(SingleImages_df, test_size = 0.10, random_state = 42, shuffle = True)
train2, valid2 = train_test_split(train2, test_size = 0.10, random_state = 42, shuffle = True)


print('train data :', train2.shape)
print('test data  :', test2.shape)
print('valid data :', valid2.shape)

In [ ]:
train = np.append(train1, train2, axis=0)
test = np.append(test1, test2, axis=0)
valid = np.append(valid1, valid2, axis=0)
print("===== Final data point shape =====")
train.shape, test.shape, valid.shape

In [ ]:
# Shuffling of data points
from sklearn.utils import shuffle
for i in range(3):
    train = shuffle(train, random_state=15)
    test = shuffle(test, random_state=15)
    valid = shuffle(valid, random_state=15)

## **BaseLine Model**

#### **Load Structured data**

In [ ]:
train = train[:-20:]      # Removing 20 samples
train.shape

In [ ]:
validation = valid[:-9:]
validation.shape             # Removing 9 samples

In [ ]:
columns = ["front X-Ray","lateral X-Ray","findings"]

train = pd.DataFrame(train,columns=columns)
test = pd.DataFrame(test,columns=columns)
validation = pd.DataFrame(validation,columns = columns)

### **Decoder**

In [ ]:
def decoder_io(df):
  '''This function Add start and end token in finding feature and prepare decoder input with output
  It takes dataframe(df) as parameter and returns df after applying necessary operations. '''
  
  df['dec_ip'] = '<start>' + ' ' + df.findings.astype(str)  #Decoder input
  df['dec_op'] = df.findings.astype(str) + ' ' +'<end>'     #Decoder output
  df['findings']      = '<start> ' + df.findings + ' <end>'
  return df

In [ ]:
train = decoder_io(train)
validation = decoder_io(validation)
test = decoder_io(test)

In [ ]:
train.head()

### **Transfer Learning : CheXNET Model  (pretrained)**

CheXNET Model is a Denset121 layered model which is trained on 112,120 number of chest x-ray images for the classification of 14 diseases. We can load the weights of that model and pass the image through that model. The top layer will be ignored.

In [ ]:
# Loading CheXnet weights

image_model =  DenseNet121(weights='/content/drive/MyDrive/B.E. Project Resources/CheXNet_weights.h5',classes=14,input_shape=(256,256,3))

In [ ]:
model = Model(image_model.input,image_model.layers[-2].output)

#### **Image to tensor convert** (Input)

In [ ]:
def load_image(img_name):
  # Loads image in array format
  image = Image.open(img_name)    # loads image into variable image
  X = np.asarray(image.convert('RGB'))    # convert function converts image into RGB format and asarray convert image into array 
  X = np.asarray(X)
  X = preprocess_input(X)
  X = resize(X,(256,256,3))
  X = np.expand_dims(X,axis = 0)
  X = np.asarray(X)

  return X 

In [ ]:
def image(df):
  '''
    input -- dataframe(df)
    output -- dataframe(df)
    process - convert images into 256 X 256, then using CHeXNET model generate tensor(concate two image tensor)

  '''
  path = '/content/images/'           # image path
  image1_paths = df['front X-Ray'].astype(str).tolist()       # Ex..['CXR1212_IM-0143-2001.png']
  image2_paths = df['lateral X-Ray'].astype(str).tolist()     # Ex..['CXR1212_IM-0143-1001.png']

  image_features = []                 # Creating a list
  for i in tqdm(range(len(image1_paths))):         #   range(len(['CXR1212_IM-0143-2001.png']))  len = 1

    i1 = load_image(path+image1_paths[i])     # i1 = load_image('/content/image/'+'CXR1212_IM-0143-2001.png'-->(image1_path[0]))
    i2 = load_image(path+image2_paths[i])     # i2 = load_image('/content/image/'+'CXR1212_IM-0143-1001.png'-->(image2_path[0]))
    img1_features = model.predict(i1)   #   img1_features = here model will extract images features using cheXNET model ,so 
    img2_features = model.predict(i2)   #   CheXNET will predict the image features 
    img1_features = np.vstack(img1_features).astype(np.float)   # vstack -- vertical stacking [123],[345] --- [[123],[345]]
    img2_features = np.vstack(img2_features).astype(np.float)   # hstack -- horizontal stacking [123],[345] -- [123345]
    
    tensor = np.concatenate((img1_features, img2_features), axis=1)   # concatenate image features 

    image_features.append(tensor)
  df['image_features'] =image_features
  return df

In [ ]:
cwd = os.getcwd()

In [ ]:
import tarfile
images = tarfile.open('/content/drive/MyDrive/B.E. Project Resources/NLMCXR_png.tgz')
images.extractall(cwd+'/images/')

In [ ]:
train.head()

In [ ]:
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

In [ ]:
train = image(train)
test = image(test)
validation = image(validation)

#train = pd.read_csv('/content/drive/MyDrive/B.E. Project Resources/train.csv',index_col=False)
#test = pd.read_csv('/content/drive/MyDrive/B.E. Project Resources/test.csv',index_col=False)
#validation = pd.read_csv('/content/drive/MyDrive/B.E. Project Resources/validation.csv',index_col=False)
#train = train.iloc[:,1:]
#test = test.iloc[:,1:]
#validation = validation.iloc[:,1:]

In [ ]:
#train.to_csv('/content/drive/MyDrive/B.E. Project Resources/train.csv')
#test.to_csv('/content/drive/MyDrive/B.E. Project Resources/test.csv')
#validation.to_csv('/content/drive/MyDrive/B.E. Project Resources/validation.csv')
np.save('/content/drive/MyDrive/B.E. Project Resources/train.npy', train)
np.save('/content/drive/MyDrive/B.E. Project Resources/test.npy', test)
np.save('/content/drive/MyDrive/B.E. Project Resources/validation.npy', validation)

In [ ]:
train_np = train.to_numpy()
test_np = test.to_numpy()
validation_np = validation.to_numpy()

np.save('train', train_np)
np.save('test', test_np)
np.save('validation', validation_np)

In [ ]:
train.head()

#### **Text Tokenization**

In [ ]:

token = Tokenizer( filters='!"#$%&()*+,-/:;=?@[\\]^_`{|}~\t\n')
train_findings = train['findings'].astype("str")
token.fit_on_texts(train_findings)


token.word_index['<pad>'] = 0
token.index_word[0] = '<pad>'
vocab_size = len(token.word_index) + 1
print('Vocab size - ', vocab_size)

#sequence in train and validation
train_inp_dec = token.texts_to_sequences(train.dec_ip)
train_op_dec = token.texts_to_sequences(train.dec_op)
val_inp_dec = token.texts_to_sequences(validation.dec_ip)
val_op_dec = token.texts_to_sequences(validation.dec_op)

#padding in the train and validation 
max_len = 100
decoder_input = pad_sequences(train_inp_dec, maxlen=max_len, padding='post')
decoder_output =  pad_sequences(train_op_dec, maxlen=max_len, padding='post') 
Validation_decoder_input = pad_sequences(val_inp_dec, maxlen=max_len, padding='post') 
Validation_decoder_output = pad_sequences(val_op_dec, maxlen=max_len, padding='post')
print(decoder_input[:1])

word_idx = {}
idx_word = {}
for key, value in (token.word_index).items(): 
    word_idx[key] = value
    idx_word[value] = key

### **Encoder and Decoder** :

 **Writing Encoder**
  :

In [ ]:
class Encoder(tf.keras.Model):
    '''
    Encoder model -- That takes a input sequence and returns encoder-outputs,encoder_final_state_h,encoder_final_state_c
    '''

    def __init__(self,lstm_units):
        super().__init__()
        self.lstm_units = lstm_units
        self.dense      = Dense(self.lstm_units,kernel_initializer = tf.keras.initializers.glorot_uniform(seed = 40),name = 'encoder_dense_layer')
        self.dropout    = Dropout(0.3)


    def call(self,x):
        
        # x : image_data
        self.encoder_output = self.dense(x)
        self.encoder_output =self.dropout(self.encoder_output )

        return self.encoder_output

**Writing Decoder**


In [ ]:
class Decoder(Model):
    '''
    Decoder model -- That takes a input sequence and returns output sequence
    '''

    def __init__(self,vocab_size,embedding_dim,lstm_units):
        super().__init__()
        self.vocab_size    = vocab_size
        self.lstm_units    = lstm_units
        self.embedding_dim = embedding_dim
        
        self.dropout   = Dropout(0.3)
        self.embedding = Embedding(self.vocab_size,self.embedding_dim,trainable = True)
        self.lstm      = LSTM(self.lstm_units, return_sequences=True, name="Decoder_LSTM")

        
    def call(self,x):
        
        # x : input_sequence
        self.target_embedd = self.embedding(x)
        self.target_embedd = self.dropout(self.target_embedd )
        self.lstm_output   = self.lstm(self.target_embedd)

        return self.lstm_output

**Encoder Decoder Model**


In [ ]:
class Encoder_decoder(Model):
    '''
        A. Pass the input sequence to Encoder layer -- Return encoder_output,encoder_final_state_h,encoder_final_state_c
        B. Pass the target sequence to Decoder layer with intial states as encoder_final_state_h,encoder_final_state_C
        C. Pass the decoder_outputs into Dense layer 
        
    '''
    def __init__(self,vocab_size,embedding_dim,lstm_units):
        super().__init__()
        
        self.vocab_size    = vocab_size
        self.lstm_units    = lstm_units
        self.embedding_dim = embedding_dim
        
        self.encoder = Encoder(self.lstm_units)
        self.decoder = Decoder(vocab_size, embedding_dim, lstm_units)

        self.dense   = TimeDistributed(Dense(self.vocab_size, activation='softmax'))
        self.dropout = Dropout(0.3)
    
  
    def call(self,data):
    
        self.input1,self.input2 = data[0], data[1]

        self.encoder_output = self.encoder(self.input1)
        self.decoder_output = self.decoder(self.input2)
        
        self.add    = tf.keras.layers.Add()([self.encoder_output, self.decoder_output])
        self.concat = self.dropout(self.add)
        output      = self.dense(self.concat)

        return output

**Load Encoder Decoder Model**

In [ ]:
lstm_units    = 512
embedding_dim = 300
tf.keras.backend.clear_session()
#This will clear all tensorflow session
Basic_model = Encoder_decoder(vocab_size,embedding_dim,lstm_units)
Basic_model.compile(optimizer=tf.keras.optimizers.Adam(0.001),loss='sparse_categorical_crossentropy')

**Replacing image tensors for training**


In [ ]:
train_image_features = np.vstack(train.image_features).astype(np.float)
validation_image_features = np.vstack(validation.image_features).astype(np.float)

**Call backs**

In [ ]:
%load_ext tensorboard

In [ ]:
!rm -rf ./logs/

In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss',  patience = 10, baseline=None, verbose = 1, restore_best_weights=True)
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.1, mode = 'min',verbose = 1, patience=5, min_lr=0.000001)
log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)

**Model Training**

In [ ]:
Basic_model.fit([train_image_features, decoder_input], decoder_output, validation_data=([validation_image_features, Validation_decoder_input], Validation_decoder_output),
                batch_size=25, epochs=50, callbacks=[early_stop,reduce_lr,tensorboard]) #epoch chamged from 100 to 50

In [ ]:
Basic_model.save_weights('Basic_model.h5')
!cp Basic_model.h5 "drive/My Drive/"

In [ ]:
Basic_model.save("Basic_model")
!cp -r Basic_model "drive/My Drive/"
Basic_model = tf.keras.models.load_model("/content/Basic_model")

**Tensorboard**

In [ ]:
%tensorboard --logdir logs/fit

**Predictions**

In [ ]:
def preprocess(image1_paths,image2_paths):

  '''
    input -- dataframe(df)
    output -- dataframe(df)
    process - convert images into 256 X 256, then using CHeXNET model generate tensor(concate two image tensor)
  
  '''
  
  path = '/content/images/'

  image_features = []
  for i in range(len(image1_paths)):

    i1 = load_image(path+image1_paths)
    i2 = load_image(path+image2_paths)
    img1_features = model.predict(i1)
    img2_features = model.predict(i2)
    img1_features = np.vstack(img1_features).astype(np.float)
    img2_features = np.vstack(img2_features).astype(np.float)
    
    tensor = np.concatenate((img1_features, img2_features), axis=1)

  return tensor

In [ ]:
def evaluate(image1, image2):
    '''
    Input - two image and image path
    output - return medical report of the images
    This function takes images and using encoder decoder model
    return medical report of the images
    
    '''

    img_tensor = preprocess(image1, image2)
    image_features = np.vstack(img_tensor).astype(np.float)
  
    encoder_output= Basic_model.layers[0](image_features) 
    encoder_output = Basic_model.layers[0].dropout(encoder_output)

    # Here replace with index of <start> in the english vocab
    input_sequence = np.ones((1, 1)) * word_idx['<start>'] 
    predicted_senence = "<start>"
    
    for i in range(25):
        decoder_output = Basic_model.layers[1].embedding(input_sequence)
        decoder_output = Basic_model.layers[1].dropout(decoder_output)
        decoder_output = Basic_model.layers[1].lstm(decoder_output)

        output = tf.keras.layers.Add()([encoder_output, decoder_output])
        output = Basic_model.layers[2](output)

        input_sequence    = np.reshape(np.argmax(output), (1, 1))
        predicted_senence = predicted_senence + ' ' + idx_word[input_sequence[0][0]+1]

        if(input_sequence[0][0] == word_idx['<end>'] ):
            break

    return predicted_senence.strip()

**Inference : Model Evaluation & Image Plotting**

In [ ]:
def test_img_cap(img_data):
    
    '''
    input - imagedata point contain two x ray image and acutal medical report of the images
    output - function return two images and its original and predical medical report
    also return bleu score of the context
    
    '''
    path = '/content/images/'
    result = evaluate(img_data[0],img_data[1]) 
    
    fig, axs = plt.subplots(1, 2, figsize = (10,10), tight_layout=True)
    count = 0
    for img, subplot in zip(img_data[:2], axs.flatten()):
        img_= mpimg.imread(path+img)
        imgplot = axs[count].imshow(img_, cmap = 'bone')
        count +=1
    plt.show()

    print('Acutal :', img_data[2])
    print("Predicted:",result)
    print('BLEU Score :-',sentence_bleu([img_data[2].split()], result.split()))     #added [] to img_data[2] and split() to both
    rouge = Rouge()
    r = rouge.get_scores(result, img_data[2], avg=True)
    print("ROUGE Score : ", r)

**Checking predictions for some random datapoints of test data**

In [ ]:
test_img_cap(test.values[105])

In [ ]:
test_img_cap(test.values[93])

In [ ]:
test_img_cap(test.values[128])

In [ ]:
test_img_cap(test.values[39])

In [ ]:
test_img_cap(test.values[66])

In [ ]:
test_img_cap(test.values[18])

In [ ]:
test_img_cap(test.values[87])

**Evaluation Metric : BLEU Score**

In [ ]:
b1 = 0
b2 = 0
b3 = 0
b4 = 0


for i in range(test.shape[0]):
  image_1 = test['front X-Ray'][i]
  image_2 = test['lateral X-Ray'][i]
  result = evaluate(image_1,image_2)
  reference = test.findings[i]
  

  
  b1 =  b1 + bleu.sentence_bleu([reference.split()], result.split() ,weights=(1, 0, 0, 0))
  b2 =  b2 + bleu.sentence_bleu([reference.split()], result.split() ,weights=(0.5, 0.5, 0, 0))
  b3 =  b3 + bleu.sentence_bleu([reference.split()], result.split() ,weights=(0.33, 0.33, 0.33, 0))
  b4 =  b4 + bleu.sentence_bleu([reference.split()], result.split() ,weights=(0.25, 0.25, 0.25, 0.25))

  rouge = Rouge()
  r = rouge.get_scores(result, reference, avg=True)


print("Bleu1 score is : ",b1/test.shape[0])
print("Bleu2 score is : ",b2/test.shape[0])
print("Bleu3 score is : ",b3/test.shape[0])
print("Bleu4 score is : ",b4/test.shape[0])

print("\n")
print("ROUGE score is : ", r)

In [ ]:
b1 = 0
b2 = 0
b3 = 0
b4 = 0


for i in range(validation.shape[0]):
  image_1 = validation['front X-Ray'][i]
  image_2 = validation['lateral X-Ray'][i]
  result = evaluate(image_1,image_2)
  reference = validation.findings[i]
  

  
  b1 =  b1 + bleu.sentence_bleu([reference.split()], result.split() ,weights=(1, 0, 0, 0))
  b2 =  b2 + bleu.sentence_bleu([reference.split()], result.split() ,weights=(0.5, 0.5, 0, 0))
  b3 =  b3 + bleu.sentence_bleu([reference.split()], result.split() ,weights=(0.33, 0.33, 0.33, 0))
  b4 =  b4 + bleu.sentence_bleu([reference.split()], result.split() ,weights=(0.25, 0.25, 0.25, 0.25))

  rouge = Rouge()
  r = rouge.get_scores(result, reference, avg=True)


print("Bleu1 score is : ",b1/validation.shape[0])
print("Bleu2 score is : ",b2/validation.shape[0])
print("Bleu3 score is : ",b3/validation.shape[0])
print("Bleu4 score is : ",b4/validation.shape[0])

print("\n")
print("ROUGE score is : ", r)